# Speaker Diarization + Multi-Model Transcription Comparison

**Optimized pipeline** with interactive dashboard to compare transcription models.

**Pipeline:** Audio → HP + Adaptive VAD + Norm → Diarize (pyannote) → Transcribe (3 models) → Merge → Dashboard

**Models compared:**
1. **Groq API** — Whisper large-v3 on cloud LPU (free, fastest)
2. **faster-whisper** — large-v3 int8 locally on CPU (free, offline)
3. **sam8000 fine-tuned** — Bulgarian-specific model (best WER 9.97%)

## 1. Setup

In [1]:
import base64
import gc
import json
import os
import re
import subprocess
import time
from pathlib import Path

import numpy as np
import torch
from scipy.io import wavfile
from scipy.signal import butter, sosfilt

import noisereduce as nr
import pyloudnorm as pyln
from groq import Groq
import anthropic

# Fix ffmpeg
try:
    subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
except (subprocess.CalledProcessError, OSError):
    os.environ["PATH"] = "/usr/bin:" + os.environ.get("PATH", "")

# Tokens
GROQ_KEY = "YOUR_GROQ_API_KEY"
HF_TOKEN = "YOUR_HF_TOKEN"
os.environ["HF_TOKEN"] = HF_TOKEN
ANTHROPIC_KEY = "YOUR_ANTHROPIC_API_KEY"

# Paths
PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output" / "model_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_FILE = DATA_DIR / "234602045" / "pe202410_real_100774_234602045_20241027_200420_71.mp4"
WAV_RAW = OUTPUT_DIR / "raw_6to12min.wav"
WAV_FILTERED = OUTPUT_DIR / "filtered_6to12min.wav"
SR = 16000
EXTRACT_START = 360   # 6:00
EXTRACT_DURATION = 360  # 6 minutes (6:00 to 12:00)

print(f"Video: {VIDEO_FILE.name} ({VIDEO_FILE.stat().st_size / 1024 / 1024:.0f} MB)")
print(f"Extract: {EXTRACT_START//60}:00 to {(EXTRACT_START+EXTRACT_DURATION)//60}:00 ({EXTRACT_DURATION}s)")
print(f"Groq key: ...{GROQ_KEY[-4:]}")
print(f"Anthropic key: ...{ANTHROPIC_KEY[-4:]}")

Video: pe202410_real_100774_234602045_20241027_200420_71.mp4 (1053 MB)
Extract: 6:00 to 12:00 (360s)
Groq key: ...de3Z
Anthropic key: ...tgAA


## 2. Audio Extraction + Optimized Preprocessing

Pipeline: **High-pass 80Hz → Adaptive VAD (gentle NR on speech, silence gaps) → Loudness norm**

In [2]:
# Extract audio from min 6 to min 12
if not WAV_RAW.exists():
    print(f"Extracting min {EXTRACT_START//60}-{(EXTRACT_START+EXTRACT_DURATION)//60} from {VIDEO_FILE.name}...")
    subprocess.run(
        ["ffmpeg", "-ss", str(EXTRACT_START), "-i", str(VIDEO_FILE),
         "-t", str(EXTRACT_DURATION),
         "-vn", "-acodec", "pcm_s16le", "-ar", str(SR), "-ac", "1",
         "-y", str(WAV_RAW)],
        capture_output=True, check=True)
    print(f"Saved: {WAV_RAW.name} ({WAV_RAW.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    print(f"WAV exists: {WAV_RAW.name}")

_, audio_int16 = wavfile.read(str(WAV_RAW))
audio_raw = audio_int16.astype(np.float32) / 32768.0
print(f"Duration: {len(audio_raw)/SR:.1f}s | Samples: {len(audio_raw):,} | SR: {SR}")

# --- Smart Hybrid Filtering ---
# Multi-Band AGC for normal sections (boosts quiet background speakers)
# Raw audio for very faint sections (AGC destroys ultra-quiet speech)
# Decided per 1-second window based on RMS level

from scipy.signal import butter, sosfilt

def agc(audio_data, sr, window_ms=500, target_rms=0.08):
    window_samples = int(sr * window_ms / 1000)
    output = np.zeros_like(audio_data)
    for i in range(0, len(audio_data), window_samples // 2):
        end = min(i + window_samples, len(audio_data))
        chunk = audio_data[i:end]
        rms = np.sqrt(np.mean(chunk**2))
        if rms > 0.001:
            gain = min(target_rms / rms, 10.0)
        else:
            gain = 1.0
        for j in range(i, end):
            if output[j] == 0: output[j] = audio_data[j] * gain
            else: output[j] = (output[j] + audio_data[j] * gain) / 2
    return np.clip(output, -1.0, 1.0)

# Build Multi-Band AGC version
band_lo = sosfilt(butter(4, [80, 300], btype="bandpass", fs=SR, output="sos"), audio_raw)
band_mid = sosfilt(butter(4, [300, 3500], btype="bandpass", fs=SR, output="sos"), audio_raw)
band_hi = sosfilt(butter(4, [3500, 7500], btype="bandpass", fs=SR, output="sos"), audio_raw)
audio_agc = agc(band_lo, SR, 500, 0.02) + agc(band_mid, SR, 300, 0.07) + agc(band_hi, SR, 300, 0.03)

# Smart blend: for each 1s window, use raw if RMS < threshold (faint speech),
# otherwise use Multi-Band AGC (catches background speakers better)
RMS_THRESHOLD = 0.015  # below this, speech is too faint for AGC
audio_f = np.zeros_like(audio_raw)
raw_secs = []
agc_secs = []

for i in range(0, len(audio_raw), SR):
    end = min(i + SR, len(audio_raw))
    rms = np.sqrt(np.mean(audio_raw[i:end]**2))
    sec = i // SR
    if rms < RMS_THRESHOLD:
        # Very quiet: use raw (AGC would destroy faint speech)
        audio_f[i:end] = audio_raw[i:end] * 8.0  # gentle boost only
        raw_secs.append(sec)
    else:
        # Normal: use Multi-Band AGC
        audio_f[i:end] = audio_agc[i:end]
        agc_secs.append(sec)

audio_f = np.clip(audio_f, -1, 1)

# Loudness normalize
meter = pyln.Meter(SR)
loudness = meter.integrated_loudness(audio_f)
audio_f = pyln.normalize.loudness(audio_f, loudness, -16.0)

wavfile.write(str(WAV_FILTERED), SR, (np.clip(audio_f, -1, 1) * 32767).astype(np.int16))
AUDIO_DURATION = len(audio_f) / SR

print(f"Filtered (Smart Hybrid): {AUDIO_DURATION:.1f}s")
print(f"  Raw+boost sections (faint speech): {len(raw_secs)} seconds — {raw_secs[:10]}{'...' if len(raw_secs)>10 else ''}")
print(f"  Multi-Band AGC sections: {len(agc_secs)} seconds")
print(f"  Saved: {WAV_FILTERED.name}")

Extracting min 6-12 from pe202410_real_100774_234602045_20241027_200420_71.mp4...


Saved: raw_6to12min.wav (11.0 MB)
Duration: 360.0s | Samples: 5,760,000 | SR: 16000


Filtered (Smart Hybrid): 360.0s
  Raw+boost sections (faint speech): 152 seconds — [0, 1, 9, 10, 18, 19, 22, 23, 26, 28]...
  Multi-Band AGC sections: 208 seconds
  Saved: filtered_6to12min.wav


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


## 3. Speaker Diarization (pyannote 3.1)

In [3]:
from pyannote.audio import Pipeline

print("Loading pyannote...")
t0 = time.time()
diar_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF_TOKEN)
diar_pipeline.to(torch.device("cpu"))
print(f"Loaded in {time.time()-t0:.1f}s")

# IMPORTANT: Diarize on RAW audio (not filtered!)
# Filtering attenuates non-speech and smooths speaker features, hurting diarization.
# We only use filtered audio for transcription.
waveform_raw = torch.from_numpy(audio_raw).float().unsqueeze(0)

print("Running diarization on RAW audio (better speaker features)...")
t0 = time.time()

# min_speakers=4 because we know there are 4+ people
diar_result = diar_pipeline(
    {"waveform": waveform_raw, "sample_rate": SR},
    min_speakers=4,
)
annotation = diar_result.speaker_diarization
print(f"Done in {time.time()-t0:.1f}s")

diar_segments = []
for turn, _, speaker in annotation.itertracks(yield_label=True):
    diar_segments.append({"speaker": speaker, "start": round(turn.start, 3), "end": round(turn.end, 3)})

speakers = sorted(set(s["speaker"] for s in diar_segments))
print(f"\nSpeakers: {len(speakers)} — {', '.join(speakers)}")
for sp in speakers:
    segs = [s for s in diar_segments if s["speaker"] == sp]
    dur = sum(s["end"] - s["start"] for s in segs)
    print(f"  {sp}: {len(segs)} segments, {dur:.1f}s")

# Release diarization model
del diar_pipeline, waveform_raw
gc.collect()

/home/airsrg/mambaforge/lib/python3.10/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.9.1+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             tabl

Loading pyannote...


[NeMo W 2026-03-29 16:23:17 <frozen importlib:241] Megatron num_microbatches_calculator not found, using Apex version.


OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.


No exporters were provided. This means that no telemetry data will be collected.


Loaded in 11.9s
Running diarization on RAW audio (better speaker features)...


Found only 2 clusters. Using a smaller value than 12 for `min_cluster_size` might help.


Done in 405.4s

Speakers: 2 — SPEAKER_00, SPEAKER_01
  SPEAKER_00: 14 segments, 34.4s
  SPEAKER_01: 62 segments, 74.3s


28

## 4. Transcription — 3 Models

In [4]:
model_results = {}

# === THREE-PASS TRANSCRIPTION ===
print("=== Three-Pass Transcription ===")
groq_client = Groq(api_key=GROQ_KEY)
from scipy.io import wavfile as wf

# --- PASS 1: Full filtered file ---
print("\nPASS 1: Full filtered file...")
t0_total = time.time()
with open(WAV_FILTERED, "rb") as f:
    resp = groq_client.audio.transcriptions.create(
        file=(WAV_FILTERED.name, f.read()),
        model="whisper-large-v3", language="bg",
        response_format="verbose_json", temperature=0.0)

pass1_segs = [{"start": s["start"], "end": s["end"], "text": s["text"].strip()}
              for s in resp.model_dump().get("segments", [])]
p1c = sum(len(s["text"]) for s in pass1_segs)
print(f"  {len(pass1_segs)} segs, {p1c} chars")

# --- PASS 2: Sparse detection + multi-technique ---
print("\nPASS 2: Sparse region multi-technique recovery...")
sparse_regions = []
for seg in pass1_segs:
    dur = seg["end"] - seg["start"]
    if dur >= 3.0 and len(seg["text"]) / dur < 5:
        sparse_regions.append(seg)

pass2_segs = []
for si, sparse in enumerate(sparse_regions):
    region_end = sparse["end"]
    for sec in range(int(sparse["start"]), int(sparse["end"])):
        s1, s2 = sec * SR, min((sec + 1) * SR, len(audio_raw))
        if np.sqrt(np.mean(audio_raw[s1:s2]**2)) > 0.06 and sec > sparse["start"]:
            region_end = float(sec)
            break

    if region_end - sparse["start"] < 1.0:
        continue

    s_start = int(max(0, sparse["start"] - 0.5) * SR)
    s_end = int(min(AUDIO_DURATION, region_end + 0.5) * SR)
    raw_chunk = audio_raw[s_start:s_end].copy()
    extract_offset = max(0, sparse["start"] - 0.5)
    rms = np.sqrt(np.mean(raw_chunk**2))

    # 4 techniques
    candidates = {}
    candidates["raw08"] = raw_chunk * (0.08 / max(rms, 1e-6))
    candidates["raw12"] = raw_chunk * (0.12 / max(rms, 1e-6))

    band = sosfilt(butter(4, [200, 4000], btype="bandpass", fs=SR, output="sos"), raw_chunk)
    ws = int(SR * 0.2)
    out = np.zeros_like(band)
    for ii in range(0, len(band), ws // 2):
        e = min(ii + ws, len(band))
        c = band[ii:e]
        r = np.sqrt(np.mean(c**2))
        g = min(0.06 / r, 12.0) if r > 0.0005 else 1.0
        for jj in range(ii, e):
            out[jj] = band[jj] * g if out[jj] == 0 else (out[jj] + band[jj] * g) / 2
    candidates["sbagc"] = np.clip(out, -1, 1)

    out2 = np.zeros_like(raw_chunk)
    for ii in range(0, len(raw_chunk), ws // 2):
        e = min(ii + ws, len(raw_chunk))
        c = raw_chunk[ii:e]
        r = np.sqrt(np.mean(c**2))
        g = min(0.06 / r, 12.0) if r > 0.0005 else 1.0
        for jj in range(ii, e):
            out2[jj] = raw_chunk[jj] * g if out2[jj] == 0 else (out2[jj] + raw_chunk[jj] * g) / 2
    candidates["agc"] = np.clip(out2, -1, 1)

    best_chars, best_key, best_resp = 0, "raw08", None
    for ck, cd in candidates.items():
        cd = np.clip(cd, -1, 1)
        tp = OUTPUT_DIR / "sparse_t.wav"
        wf.write(str(tp), SR, (cd * 32767).astype(np.int16))
        with open(tp, "rb") as f:
            cr = groq_client.audio.transcriptions.create(
                file=("t.wav", f.read()), model="whisper-large-v3",
                language="bg", response_format="verbose_json", temperature=0.0)
        ct = cr.model_dump().get("text", "")
        if len(ct) > best_chars:
            best_chars, best_key, best_resp = len(ct), ck, cr
        tp.unlink(missing_ok=True)

    sp_start = sparse["start"]
    print(f"  Sparse [{sp_start:.0f}-{region_end:.0f}s]: best={best_key} ({best_chars} chars)")
    if best_resp:
        for s in best_resp.model_dump().get("segments", []):
            mid = s["start"] + extract_offset + (s["end"] - s["start"]) / 2
            if sparse["start"] <= mid <= region_end:
                pass2_segs.append({"start": s["start"] + extract_offset,
                                   "end": s["end"] + extract_offset, "text": s["text"].strip()})

# --- PASS 3: Boosted-VAD recovery ---
print("\nPASS 3: Boosted-VAD recovery of faint speech...")
audio_5x = np.clip(audio_raw * 5.0, -1, 1)
vad_model2, vad_utils2 = torch.hub.load("snakers4/silero-vad", "silero_vad", trust_repo=True)
get_speech_ts2 = vad_utils2[0]

vad_ts = get_speech_ts2(torch.from_numpy(audio_5x).float(), vad_model2,
    sampling_rate=SR, threshold=0.3, min_speech_duration_ms=300,
    min_silence_duration_ms=400, speech_pad_ms=300)

pass3_segs = []
existing_segs = pass1_segs + pass2_segs
for ts in vad_ts:
    ts_start, ts_end = ts["start"] / SR, ts["end"] / SR
    ts_dur = ts_end - ts_start
    if ts_dur < 0.5:
        continue
    chars_here = 0
    for s in existing_segs:
        ov_s, ov_e = max(s["start"], ts_start), min(s["end"], ts_end)
        if ov_e > ov_s:
            chars_here += len(s["text"]) * (ov_e - ov_s) / (s["end"] - s["start"])
    if chars_here / max(ts_dur, 0.1) >= 3:
        continue

    chunk = audio_raw[ts["start"]:ts["end"]].copy()
    chunk_rms = np.sqrt(np.mean(chunk**2))
    if chunk_rms < 0.025:
        band = sosfilt(butter(4, [200, 4000], btype="bandpass", fs=SR, output="sos"), chunk)
        ws3 = int(SR * 0.2)
        out3 = np.zeros_like(band)
        for ii in range(0, len(band), ws3 // 2):
            e = min(ii + ws3, len(band))
            c = band[ii:e]
            r = np.sqrt(np.mean(c**2))
            g = min(0.07 / r, 12.0) if r > 0.0003 else 1.0
            for jj in range(ii, e):
                out3[jj] = band[jj] * g if out3[jj] == 0 else (out3[jj] + band[jj] * g) / 2
        chunk = np.clip(out3, -1, 1)
    else:
        chunk = np.clip(chunk * (0.08 / max(chunk_rms, 1e-6)), -1, 1)

    tp = OUTPUT_DIR / "vad_t.wav"
    wf.write(str(tp), SR, (chunk * 32767).astype(np.int16))
    with open(tp, "rb") as f:
        vr = groq_client.audio.transcriptions.create(
            file=("v.wav", f.read()), model="whisper-large-v3",
            language="bg", response_format="verbose_json", temperature=0.0)
    tp.unlink(missing_ok=True)
    text = vr.model_dump().get("text", "").strip()
    offset = ts["start"] / SR
    for s in vr.model_dump().get("segments", []):
        pass3_segs.append({"start": s["start"] + offset, "end": s["end"] + offset, "text": s["text"].strip()})
    if text:
        print(f"  [{ts_start:>5.1f}-{ts_end:>5.1f}s] RMS={chunk_rms:.4f}: {text[:70]}")

# MERGE
all_segs = [{"start": s["start"], "end": s["end"], "text": s["text"]}
            for s in pass1_segs + pass2_segs + pass3_segs]
all_segs.sort(key=lambda x: x["start"])
groq_time = time.time() - t0_total

p2c = sum(len(s["text"]) for s in pass2_segs)
p3c = sum(len(s["text"]) for s in pass3_segs)
model_results["groq_large_v3"] = {
    "name": "Groq API (three-pass)", "segments": all_segs,
    "time": groq_time, "type": "cloud"}
print(f"\nMERGED: {len(all_segs)} segs | P1:{p1c} + P2:{p2c} + P3:{p3c} = {p1c+p2c+p3c} chars in {groq_time:.1f}s")

=== Three-Pass Transcription ===

PASS 1: Full filtered file...


  118 segs, 2151 chars

PASS 2: Sparse region multi-technique recovery...


  Sparse [25-29s]: best=raw08 (55 chars)


  Sparse [55-72s]: best=agc (110 chars)


  Sparse [85-95s]: best=agc (158 chars)


...

  Sparse [330-341s]: best=agc (83 chars)


  Sparse [357-360s]: best=sbagc (24 chars)

PASS 3: Boosted-VAD recovery of faint speech...


Using cache found in /home/airsrg/.cache/torch/hub/snakers4_silero-vad_master


  [263.6-269.9s] RMS=0.0138: Не е тежи, но не е върху няма възможно. Не е върху няма. Не е върху ня


  [269.9-273.3s] RMS=0.0084: Здравейте, дете.

MERGED: 168 segs | P1:2151 + P2:791 + P3:87 = 3029 chars in 146.4s


In [5]:
# Local model switched off to save time
# Groq two-pass is the primary transcription model
print(f"Models ready: {len(model_results)}")
for key, r in model_results.items():
    chars = sum(len(s["text"]) for s in r["segments"])
    print(f"  {r['name']}: {len(r['segments'])} segs, {chars} chars, {r['time']:.1f}s")

Models ready: 1
  Groq API (three-pass): 168 segs, 3029 chars, 146.4s


## 5. Merge Diarization + Transcripts, Split Audio by Speaker

In [6]:
def merge_diarization(diar_segs, whisper_segs):
    """Assign speaker labels to transcript segments by timestamp midpoint overlap."""
    merged = []
    for ws in whisper_segs:
        w_mid = (ws["start"] + ws["end"]) / 2
        best_speaker, best_dist = None, float("inf")
        for ds in diar_segs:
            if ds["start"] <= w_mid <= ds["end"]:
                best_speaker = ds["speaker"]; break
            dist = min(abs(w_mid - ds["start"]), abs(w_mid - ds["end"]))
            if dist < best_dist:
                best_dist = dist; best_speaker = ds["speaker"]
        merged.append({**ws, "speaker": best_speaker or "UNKNOWN"})
    return merged

# Merge for each model
merged_results = {}
for model_key, mr in model_results.items():
    merged = merge_diarization(diar_segments, mr["segments"])
    merged_results[model_key] = {**mr, "merged": merged}
    print(f"{mr['name']}: {len(merged)} merged segments")

# No speaker audio splitting — just transcripts
speaker_wavs = {}
print("Speaker splitting disabled")

Groq API (three-pass): 168 merged segments
Speaker splitting disabled


## 5a. LLM Transcript Correction

Send the raw transcript to Claude to fix transcription errors using context:
- Fix garbled Bulgarian words (e.g., "осеми" → "осем", "номери" → "номера")
- Correct misheard numbers (e.g., "осемайсет" → "осемнадесет")
- Fix grammar while preserving the original meaning
- Keep timestamps unchanged

In [7]:
# LLM Transcript Correction — fix ASR errors using context
print("=== LLM Transcript Correction ===")
client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

# Build the raw transcript for correction
best_model = "groq_large_v3"
best_merged = merged_results[best_model]["merged"]

raw_lines = []
for seg in best_merged:
    m, s = divmod(seg["start"], 60)
    raw_lines.append(f'[{int(m):02d}:{s:05.2f}] {seg["speaker"]}: {seg["text"]}')

raw_transcript = "\n".join(raw_lines)

correction_prompt = """You are an expert editor of Bulgarian election committee meeting transcripts.

The transcript below was produced by automatic speech recognition (Whisper) from a Bulgarian election committee video. It contains many errors due to:
1. Faint/quiet speech that was hard to hear
2. Overlapping speakers
3. ASR misrecognizing Bulgarian words

Your task: Fix the transcript line by line. For each line, output the CORRECTED Bulgarian text.

Rules:
- Fix garbled/misspelled Bulgarian words to the most logical correct word
- Fix misheard numbers (e.g., "осеми" → "осем", "осемайсет" → "осемнадесет")
- "номери" → "номера", "семината" → "седемнадесет" or "седем" based on context
- Remove obvious hallucinations (random English words, "Абонирайте се", etc.)
- If a line is just "..." or nonsense, output "---" to mark it as unclear
- Keep the [timestamp] SPEAKER_XX: prefix EXACTLY as is — only fix the text after the colon
- This is a vote counting session — numbers and questions like "имаме ли" (do we have), "има ли" (is there), "някъде" (somewhere) are expected
- Preserve the original meaning — don't invent content, only fix what's garbled

Return ONLY the corrected transcript, one line per original line, same format:
[MM:SS.SS] SPEAKER_XX: corrected text"""

print(f"  Sending {len(raw_lines)} lines for correction...")
t0 = time.time()

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=8192,
    messages=[{"role": "user", "content": f"Fix this Bulgarian election transcript:\n\n{raw_transcript}"}],
    system=correction_prompt,
)

correction_time = time.time() - t0
corrected_text = response.content[0].text
print(f"  Corrected in {correction_time:.1f}s")

# Parse corrected lines back into segments
import re
corrected_segs = []
for line in corrected_text.strip().split("\n"):
    line = line.strip()
    if not line:
        continue
    # Parse [MM:SS.SS] SPEAKER_XX: text
    match = re.match(r'\[(\d+):(\d+\.?\d*)\]\s*(SPEAKER_\d+):\s*(.*)', line)
    if match:
        mins, secs, speaker, text = match.groups()
        text = text.strip()
        if text and text != "---":
            start = float(mins) * 60 + float(secs)
            corrected_segs.append({"start": start, "speaker": speaker, "text": text})

# Update merged_results with corrected text
if corrected_segs:
    # Match corrected segments back to original by index (same order)
    for i, cseg in enumerate(corrected_segs):
        if i < len(best_merged):
            best_merged[i]["text_original"] = best_merged[i]["text"]
            best_merged[i]["text"] = cseg["text"]
    
    merged_results[best_model]["merged"] = best_merged
    
    # Show corrections
    changes = 0
    for seg in best_merged:
        orig = seg.get("text_original", "")
        if orig and orig != seg["text"]:
            changes += 1
    
    print(f"\n  Corrections made: {changes} out of {len(best_merged)} segments")
    print(f"\n  Sample corrections (first 15):")
    shown = 0
    for seg in best_merged:
        orig = seg.get("text_original", "")
        if orig and orig != seg["text"] and shown < 15:
            m, s = divmod(seg["start"], 60)
            print(f"    [{int(m):02d}:{s:05.2f}] BEFORE: {orig[:60]}")
            print(f"    {'':>10} AFTER:  {seg['text'][:60]}")
            print()
            shown += 1
else:
    print("  WARNING: Could not parse corrected transcript")

=== LLM Transcript Correction ===
  Sending 168 lines for correction...


  Corrected in 34.2s

  Corrections made: 156 out of 168 segments

  Sample corrections (first 15):
    [00:00.96] BEFORE: Седьам, осем, номери.
               AFTER:  Седем, осем, номера.

    [00:03.60] BEFORE: Девица.
               AFTER:  Девет.

    [00:05.28] BEFORE: Аз овсем блашам.
               AFTER:  Аз осем броя.

    [00:06.44] BEFORE: Шест, дев Video heavenly.
               AFTER:  Шест, девет.

    [00:08.88] BEFORE: Двидата.
               AFTER:  Двадесет.

    [00:12.04] BEFORE: Остройството не е позиционирано правилно.
               AFTER:  Устройството не е позиционирано правилно.

    [00:18.90] BEFORE: А това е оптината.
               AFTER:  А това е осмата.

    [00:23.60] BEFORE: Не е сгseл.
               AFTER:  Не е съвсем.

    [00:24.76] BEFORE: … нашото еляальное стрелязимо в тезиρείта идеи стел si
               AFTER:  Българската е фазна.

    [00:25.26] BEFORE: Пълската е фазни.
               AFTER:  12, 5, 4, 5, 6, 7, 8, 9, 10.

    [00:28.92] 

## 5b. Waveform Visualization — Before vs After Filtering

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Generate waveform comparison: raw vs filtered
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

time_axis = np.arange(len(audio_raw)) / SR

# Plot 1: Raw waveform
axes[0].plot(time_axis, audio_raw, color='#3498db', linewidth=0.3, alpha=0.7)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Raw Audio (before filtering)', fontsize=12, fontweight='bold')
axes[0].set_ylim(-0.3, 0.3)
axes[0].axvspan(0, 12, alpha=0.15, color='red', label='Faint speech (0-12s)')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Plot 2: Filtered waveform
axes[1].plot(time_axis, audio_f, color='#27ae60', linewidth=0.3, alpha=0.7)
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Filtered Audio (Smart Hybrid: Multi-Band AGC + raw boost for faint sections)', fontsize=12, fontweight='bold')
axes[1].set_ylim(-0.3, 0.3)
axes[1].axvspan(0, 12, alpha=0.15, color='red', label='Faint speech (boosted)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Plot 3: Per-second RMS comparison
rms_raw = []
rms_filt = []
rms_times = []
for i in range(0, len(audio_raw) - SR, SR):
    rms_raw.append(np.sqrt(np.mean(audio_raw[i:i+SR]**2)))
    rms_filt.append(np.sqrt(np.mean(audio_f[i:i+SR]**2)))
    rms_times.append(i / SR)

axes[2].bar(rms_times, rms_raw, width=0.8, alpha=0.5, color='#3498db', label='Raw RMS')
axes[2].bar(rms_times, rms_filt, width=0.4, alpha=0.7, color='#27ae60', label='Filtered RMS')
axes[2].set_ylabel('RMS Energy')
axes[2].set_xlabel('Time (seconds)')
axes[2].set_title('Per-Second RMS Energy Comparison', fontsize=12, fontweight='bold')
axes[2].axvspan(0, 12, alpha=0.15, color='red', label='Faint section')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
waveform_path = OUTPUT_DIR / "waveform_comparison.png"
fig.savefig(str(waveform_path), dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved waveform comparison: {waveform_path}")

# Also generate zoomed view of 0-15s
fig2, axes2 = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
t_zoom = np.arange(15*SR) / SR

axes2[0].plot(t_zoom, audio_raw[:15*SR], color='#3498db', linewidth=0.5)
axes2[0].set_ylabel('Amplitude')
axes2[0].set_title('0-15s Raw Audio (faint vote counting section)', fontsize=12, fontweight='bold')
axes2[0].set_ylim(-0.25, 0.25)
axes2[0].grid(True, alpha=0.3)

axes2[1].plot(t_zoom, audio_f[:15*SR], color='#27ae60', linewidth=0.5)
axes2[1].set_ylabel('Amplitude')
axes2[1].set_title('0-15s Filtered Audio (boosted faint speech)', fontsize=12, fontweight='bold')
axes2[1].set_ylim(-0.25, 0.25)
axes2[1].set_xlabel('Time (seconds)')
axes2[1].grid(True, alpha=0.3)

plt.tight_layout()
zoom_path = OUTPUT_DIR / "waveform_zoom_0to15.png"
fig2.savefig(str(zoom_path), dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved zoom view: {zoom_path}")

# Encode as base64 for dashboard
with open(waveform_path, "rb") as f:
    waveform_b64 = base64.b64encode(f.read()).decode("utf-8")
with open(zoom_path, "rb") as f:
    zoom_b64 = base64.b64encode(f.read()).decode("utf-8")

Saved waveform comparison: /home/airsrg/work/video_streaming/output/model_comparison/waveform_comparison.png


Saved zoom view: /home/airsrg/work/video_streaming/output/model_comparison/waveform_zoom_0to15.png


## 6. LLM Analysis — Speaker Roles + Conversation KPIs

Uses Claude API to analyze the full conversation and extract:
- Speaker roles (Chair, Committee Member, Observer, Device Voice)
- Tension/arguing scale (1-5)
- Vote numbers and tallies
- Names mentioned
- "Наши/Ваши" partisan language
- "Невалидни" (invalid votes) mentions

In [9]:
# Use the best model's transcript (Groq) for LLM analysis
best_model = "groq_large_v3"
best_merged = merged_results[best_model]["merged"]

# Build conversation text with timestamps
conversation_text = ""
for seg in best_merged:
    m, s = divmod(seg["start"], 60)
    conversation_text += f"[{int(m):02d}:{s:05.2f}] {seg['speaker']}: {seg['text']}\n"

# Per-speaker stats
speaker_stats = ""
for sp in speakers:
    segs = [s for s in best_merged if s["speaker"] == sp]
    dur = sum(s["end"] - s["start"] for s in segs)
    chars = sum(len(s["text"]) for s in segs)
    speaker_stats += f"  {sp}: {len(segs)} segments, {dur:.1f}s speaking time, {chars} chars\n"

# Claude API call
client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

system_prompt = """You are an expert analyst of Bulgarian election committee meeting transcripts.
Analyze the provided diarized transcript and return a JSON object with the following structure:

{
  "speaker_roles": {
    "SPEAKER_XX": {
      "role": "role name in Bulgarian (e.g., Председател, Член на комисия, Наблюдател, Глас на устройство)",
      "role_en": "role name in English",
      "confidence": 0.0-1.0,
      "reasoning": "brief explanation why"
    }
  },
  "tension_scale": {
    "overall": 1-5,
    "description": "brief description of the tension level",
    "tense_moments": ["timestamp and description of tense exchanges"]
  },
  "number_frequency_matrix": {
    "description": "Every number (1-30) mentioned in the transcript, how many times, and by whom",
    "numbers": [
      {"number": N, "total_mentions": X, "by_speaker": {"SPEAKER_XX": count}, "contexts": ["quote where it appears"]}
    ],
    "grand_total_mentions": N,
    "sum_of_all_numbers": N
  },
  "candidate_votes": {
    "tallies": [{"candidate_number": N, "times_mentioned": X, "vote_count_stated": Y}],
    "total_votes_stated": N,
    "interpretation": "what these vote numbers represent"
  },
  "names_mentioned": {
    "names": [{"name": "...", "count": X, "by_speaker": "SPEAKER_XX"}]
  },
  "partisan_language": {
    "nashi_count": X,
    "vashi_count": X,
    "per_speaker": {"SPEAKER_XX": {"nashi": X, "vashi": X}},
    "examples": ["quotes with context"]
  },
  "invalid_votes": {
    "nevalidni_count": X,
    "mentions": [{"speaker": "SPEAKER_XX", "timestamp": "...", "context": "quote"}]
  }
}

IMPORTANT: For number_frequency_matrix, list EVERY number from 1 to 30 that is mentioned in the transcript (as digits or as Bulgarian words like "едно", "две", "три", etc.). Count each mention separately. Track which speaker said each number.

Return ONLY valid JSON, no markdown or explanation outside the JSON."""

user_msg = f"""Analyze this Bulgarian election committee meeting transcript.

SPEAKER STATISTICS:
{speaker_stats}

FULL TRANSCRIPT:
{conversation_text}"""

print("Sending to Claude API for analysis...")
t0 = time.time()
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=4096,
    messages=[{"role": "user", "content": user_msg}],
    system=system_prompt,
)
llm_time = time.time() - t0
print(f"Claude response in {llm_time:.1f}s")

# Parse response
llm_text = response.content[0].text
try:
    json_match = re.search(r'\{[\s\S]*\}', llm_text)
    if json_match:
        analysis = json.loads(json_match.group())
    else:
        analysis = json.loads(llm_text)
    print("Analysis parsed successfully!")
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(f"Raw response:\n{llm_text[:500]}")
    analysis = {}

# Display results
if analysis:
    print(f"\n{'='*60}")
    print("SPEAKER ROLES")
    print(f"{'='*60}")
    for sp, info in analysis.get("speaker_roles", {}).items():
        print(f"  {sp}: {info.get('role', '?')} ({info.get('role_en', '?')}) — {info.get('reasoning', '')}")
    
    print(f"\n{'='*60}")
    print("TENSION SCALE")
    print(f"{'='*60}")
    ts = analysis.get("tension_scale", {})
    print(f"  Overall: {ts.get('overall', '?')}/5 — {ts.get('description', '')}")
    
    print(f"\n{'='*60}")
    print("NUMBER FREQUENCY MATRIX")
    print(f"{'='*60}")
    nfm = analysis.get("number_frequency_matrix", {})
    print(f"  {'Number':<10} {'Mentions':>10} {'By Speaker'}")
    print(f"  {'-'*50}")
    for entry in nfm.get("numbers", []):
        by_sp = ", ".join(f"{k}:{v}" for k, v in entry.get("by_speaker", {}).items())
        print(f"  {entry.get('number','?'):<10} {entry.get('total_mentions','?'):>10} {by_sp}")
    print(f"  Total mentions: {nfm.get('grand_total_mentions', '?')}")
    print(f"  Sum of numbers: {nfm.get('sum_of_all_numbers', '?')}")
    
    print(f"\n{'='*60}")
    print("NAMES MENTIONED")
    print(f"{'='*60}")
    for n in analysis.get("names_mentioned", {}).get("names", []):
        print(f"  {n.get('name','?')}: {n.get('count','?')}x (by {n.get('by_speaker','?')})")
    
    pl = analysis.get("partisan_language", {})
    print(f"\n  Наши: {pl.get('nashi_count', 0)}x | Ваши: {pl.get('vashi_count', 0)}x")
    
    iv = analysis.get("invalid_votes", {})
    print(f"  Невалидни: {iv.get('nevalidni_count', 0)}x")

Sending to Claude API for analysis...


Claude response in 34.3s
Analysis parsed successfully!

SPEAKER ROLES
  SPEAKER_00: Глас на устройство (Device voice) — Repeated automatic messages about device positioning and physical buttons being on top
  SPEAKER_01: Член на комисия (Commission member) — Counting votes and ballot papers, mentions writing down results, handling election materials

TENSION SCALE
  Overall: 3/5 — Moderate tension due to technical difficulties with voting device positioning and confusion during vote counting

NUMBER FREQUENCY MATRIX
  Number       Mentions By Speaker
  --------------------------------------------------
  1                   7 SPEAKER_01:7
  2                   3 SPEAKER_01:3
  3                   5 SPEAKER_01:5
  4                   3 SPEAKER_01:3
  5                   4 SPEAKER_01:4
  6                  15 SPEAKER_01:15
  7                   4 SPEAKER_01:4
  8                   7 SPEAKER_01:7
  9                   6 SPEAKER_01:6
  10                 12 SPEAKER_01:12
  11              

## 6b. Unified Transcript (Timestamp + Role + Text)

Single file combining diarization, transcription, and role identification.

In [10]:
# Build role mapping from analysis
role_map = {}
if analysis:
    for sp, info in analysis.get("speaker_roles", {}).items():
        role_map[sp] = info.get("role", sp)

# Use the best model's merged segments (Groq)
best_merged = merged_results["groq_large_v3"]["merged"]

# Offset timestamps to original video time (add EXTRACT_START)
def fmt_video_time(seconds):
    """Format as MM:SS.ss relative to original video."""
    total = seconds + EXTRACT_START
    m, s = divmod(total, 60)
    return f"{int(m):02d}:{s:05.2f}"

def fmt_srt_video(seconds):
    """Format as HH:MM:SS,mmm relative to original video."""
    total = seconds + EXTRACT_START
    h = int(total // 3600)
    m = int((total % 3600) // 60)
    s = int(total % 60)
    ms = int((total % 1) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

# --- TXT: Human-readable unified transcript ---
txt_lines = []
txt_lines.append(f"# Unified Transcript — {VIDEO_FILE.name}")
txt_lines.append(f"# Segment: {EXTRACT_START//60}:00 - {(EXTRACT_START+EXTRACT_DURATION)//60}:00")
txt_lines.append(f"# Speakers: {', '.join(f'{sp} = {role_map.get(sp, sp)}' for sp in speakers)}")
txt_lines.append(f"# Model: Groq API (Whisper large-v3)")
txt_lines.append("")

for seg in best_merged:
    role = role_map.get(seg["speaker"], seg["speaker"])
    txt_lines.append(f"[{fmt_video_time(seg['start'])} - {fmt_video_time(seg['end'])}] {role}: {seg['text']}")

txt_path = OUTPUT_DIR / "unified_transcript.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(txt_lines))

# --- SRT: Subtitle format with roles ---
srt_lines = []
for i, seg in enumerate(best_merged, 1):
    role = role_map.get(seg["speaker"], seg["speaker"])
    srt_lines.append(str(i))
    srt_lines.append(f"{fmt_srt_video(seg['start'])} --> {fmt_srt_video(seg['end'])}")
    srt_lines.append(f"{role}: {seg['text']}")
    srt_lines.append("")

srt_path = OUTPUT_DIR / "unified_transcript.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(srt_lines))

# --- JSON: Full structured output ---
json_data = {
    "video_file": VIDEO_FILE.name,
    "segment": f"{EXTRACT_START//60}:00 - {(EXTRACT_START+EXTRACT_DURATION)//60}:00",
    "speakers": {sp: role_map.get(sp, sp) for sp in speakers},
    "model": "Groq API (Whisper large-v3)",
    "analysis": analysis,
    "transcript": [
        {
            "start": seg["start"] + EXTRACT_START,
            "end": seg["end"] + EXTRACT_START,
            "speaker": seg["speaker"],
            "role": role_map.get(seg["speaker"], seg["speaker"]),
            "text": seg["text"],
        }
        for seg in best_merged
    ],
}

json_path = OUTPUT_DIR / "unified_transcript.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)

print(f"Saved unified transcript in 3 formats:")
print(f"  TXT: {txt_path.name}")
print(f"  SRT: {srt_path.name}")
print(f"  JSON: {json_path.name}")

# Print preview
print(f"\n{'='*70}")
print(f"UNIFIED TRANSCRIPT PREVIEW")
print(f"{'='*70}")
for line in txt_lines[:20]:
    print(line)
if len(txt_lines) > 20:
    print(f"... ({len(txt_lines) - 20} more lines)")
print(f"\nTotal: {len(best_merged)} segments")

Saved unified transcript in 3 formats:
  TXT: unified_transcript.txt
  SRT: unified_transcript.srt
  JSON: unified_transcript.json

UNIFIED TRANSCRIPT PREVIEW
# Unified Transcript — pe202410_real_100774_234602045_20241027_200420_71.mp4
# Segment: 6:00 - 12:00
# Speakers: SPEAKER_00 = Глас на устройство, SPEAKER_01 = Член на комисия
# Model: Groq API (Whisper large-v3)

[06:00.96 - 06:03.18] Член на комисия: Седем, осем, номера.
[06:03.60 - 06:03.78] Член на комисия: Девет.
[06:04.28 - 06:04.86] Член на комисия: Осем.
[06:05.28 - 06:06.26] Член на комисия: Аз осем броя.
[06:06.44 - 06:08.48] Член на комисия: Шест, девет.
[06:08.88 - 06:09.22] Член на комисия: Двадесет.
[06:12.04 - 06:14.48] Глас на устройство: Устройството не е позиционирано правилно.
[06:14.98 - 06:17.92] Глас на устройство: Завъртете го така, че физическите бутони да бъдат отгоре.
[06:18.90 - 06:21.76] Член на комисия: А това е осмата.
[06:23.60 - 06:25.08] Член на комисия: Не е съвсем.
[06:24.76 - 06:29.42] Член на к

## 7. Generate Interactive Dashboard

HTML dashboard with audio players, model comparison, speaker roles, and KPI cards.

In [11]:
# Speaker splitting disabled — no per-speaker WAVs
speaker_audio_b64 = {}

with open(WAV_FILTERED, "rb") as f:
    full_audio_b64 = base64.b64encode(f.read()).decode("utf-8")

# Get role labels from analysis
speaker_roles = {}
if analysis:
    for sp, info in analysis.get("speaker_roles", {}).items():
        speaker_roles[sp] = f"{info.get('role', sp)} ({info.get('role_en', '')})"

# Build dashboard data
dashboard_data = {"speakers": speakers, "models": {}, "analysis": analysis}

for model_key, mr in merged_results.items():
    model_info = {
        "name": mr["name"], "time": round(mr["time"], 1), "type": mr["type"],
        "total_chars": sum(len(s["text"]) for s in mr["merged"]),
        "total_segments": len(mr["merged"]), "per_speaker": {},
    }
    all_text = " ".join(s["text"] for s in mr["merged"])
    cyrillic = len(re.findall(r'[а-яА-ЯёЁ]', all_text))
    latin = len(re.findall(r'[a-zA-Z]', all_text))
    total = cyrillic + latin
    model_info["latin_pct"] = round((latin / total * 100) if total > 0 else 0, 1)
    for speaker in speakers:
        segs = [s for s in mr["merged"] if s["speaker"] == speaker]
        model_info["per_speaker"][speaker] = {
            "segments": segs, "text": " ".join(s["text"] for s in segs),
            "chars": sum(len(s["text"]) for s in segs), "count": len(segs),
        }
    dashboard_data["models"][model_key] = model_info

print("Dashboard data prepared")
for mk, mi in dashboard_data["models"].items():
    print(f"  {mi['name']}: {mi['total_segments']} segs, {mi['total_chars']} chars, {mi['latin_pct']}% Latin")

Dashboard data prepared
  Groq API (three-pass): 168 segs, 3027 chars, 0.0% Latin


In [12]:
import json as _json

model_keys = list(dashboard_data["models"].keys())
a = analysis or {}

# --- KPI values ---
ts = a.get("tension_scale", {})
tension_val = ts.get("overall", "?")
tension_desc = ts.get("description", "N/A")
tension_color = ["#27ae60","#27ae60","#f39c12","#e67e22","#e74c3c","#c0392b"][min(int(tension_val) if str(tension_val).isdigit() else 1, 5)]

pl = a.get("partisan_language", {})
nashi = pl.get("nashi_count", 0)
vashi = pl.get("vashi_count", 0)
iv = a.get("invalid_votes", {})
nevalidni = iv.get("nevalidni_count", 0)
nfm = a.get("number_frequency_matrix", {})
cv = a.get("candidate_votes", {})

# --- Number frequency matrix table ---
num_rows = ""
for entry in sorted(nfm.get("numbers", []), key=lambda x: x.get("number", 0)):
    n = entry.get("number", "?")
    mentions = entry.get("total_mentions", 0)
    by_sp = entry.get("by_speaker", {})
    by_sp_str = ", ".join(f"{k.replace('SPEAKER_','S')}: {v}" for k, v in by_sp.items())
    ctx = "; ".join(entry.get("contexts", [])[:2])
    bar_width = min(mentions * 30, 200)
    num_rows += f"""<tr>
      <td><strong>{n}</strong></td>
      <td>{mentions}</td>
      <td><div style="background:#3498db;height:14px;width:{bar_width}px;border-radius:3px"></div></td>
      <td style="font-size:11px;color:#666">{by_sp_str}</td>
      <td style="font-size:11px;color:#888">{ctx[:80]}</td>
    </tr>"""
if not num_rows:
    num_rows = "<tr><td colspan='5'>No numbers detected</td></tr>"

# Vote tallies
vote_rows = ""
for t in cv.get("tallies", []):
    vote_rows += f"<tr><td>#{t.get('candidate_number','?')}</td><td>{t.get('times_mentioned','?')}</td><td><strong>{t.get('vote_count_stated', t.get('vote_count','?'))}</strong></td></tr>"
if not vote_rows:
    vote_rows = "<tr><td colspan='3'>No vote data detected</td></tr>"

# Names
names_list = ""
for n in a.get("names_mentioned", {}).get("names", []):
    names_list += f"<span class='tag'>{n.get('name','?')} ({n.get('count','?')}x)</span> "
if not names_list:
    names_list = "<span style='color:#999'>No names detected</span>"

kpi_section = f'''
<div class="kpi-grid">
  <div class="kpi-card">
    <div class="kpi-label">Tension Level</div>
    <div class="kpi-value" style="color:{tension_color}">{tension_val}/5</div>
    <div class="kpi-detail">{tension_desc}</div>
  </div>
  <div class="kpi-card">
    <div class="kpi-label">Наши / Ваши</div>
    <div class="kpi-value">{nashi} / {vashi}</div>
    <div class="kpi-detail">Partisan language</div>
  </div>
  <div class="kpi-card">
    <div class="kpi-label">Невалидни</div>
    <div class="kpi-value">{nevalidni}</div>
    <div class="kpi-detail">Invalid vote mentions</div>
  </div>
  <div class="kpi-card">
    <div class="kpi-label">Numbers Mentioned</div>
    <div class="kpi-value">{nfm.get("grand_total_mentions", "?")}</div>
    <div class="kpi-detail">Total number mentions (sum: {nfm.get("sum_of_all_numbers", "?")})</div>
  </div>
</div>

<div class="section-card" style="margin-bottom:20px">
  <h3>Number Frequency Matrix (1-30)</h3>
  <table class="inner-table">
    <tr><th>Number</th><th>Mentions</th><th>Frequency</th><th>By Speaker</th><th>Context</th></tr>
    {num_rows}
  </table>
</div>

<div class="section-grid">
  <div class="section-card">
    <h3>Candidate Vote Tallies</h3>
    <table class="inner-table">
      <tr><th>Candidate</th><th>Mentions</th><th>Votes Stated</th></tr>
      {vote_rows}
    </table>
    <div style="margin-top:8px;font-size:12px;color:#666">Total votes stated: {cv.get("total_votes_stated", cv.get("total_votes", "?"))}</div>
  </div>
  <div class="section-card">
    <h3>Names Mentioned</h3>
    <div class="names-list">{names_list}</div>
  </div>
</div>'''

# --- Unified transcript HTML ---
transcript_html = ""
for seg in merged_results["groq_large_v3"]["merged"]:
    role = speaker_roles.get(seg["speaker"], seg["speaker"])
    total_s = seg["start"] + EXTRACT_START
    total_e = seg["end"] + EXTRACT_START
    m1, s1 = divmod(total_s, 60)
    m2, s2 = divmod(total_e, 60)
    sp_colors = {"SPEAKER_00":"#e74c3c","SPEAKER_01":"#2980b9","SPEAKER_02":"#27ae60","SPEAKER_03":"#f39c12","SPEAKER_04":"#8e44ad"}
    sp_color = sp_colors.get(seg["speaker"],"#333")
    transcript_html += f'<div class="tseg" data-speaker="{seg["speaker"]}"><span class="ts">[{int(m1):02d}:{s1:05.2f} - {int(m2):02d}:{s2:05.2f}]</span> <span style="color:{sp_color}"><strong>{role}:</strong></span> {seg["text"]}</div>\n'

transcript_filter_options = "".join(f'<option value="{sp}">{sp.replace("SPEAKER_","Speaker ")} - {speaker_roles.get(sp,sp)}</option>' for sp in speakers)

# --- Speaker sections with roles ---
speaker_sections = ""
for speaker in speakers:
    role_label = speaker_roles.get(speaker, speaker)
    sp_label = f'{speaker.replace("SPEAKER_", "Speaker ")} — {role_label}'
    audio_tag = ""  # Speaker audio splitting disabled
    
    model_cols = ""
    for mk in model_keys:
        mi = dashboard_data["models"][mk]
        sp_data = mi["per_speaker"].get(speaker, {"segments":[],"chars":0,"count":0})
        seg_html = ""
        for seg in sp_data["segments"]:
            m1, s1 = divmod(seg["start"], 60)
            m2, s2 = divmod(seg["end"], 60)
            seg_html += f'<div class="seg"><span class="ts">[{int(m1):02d}:{s1:05.2f}-{int(m2):02d}:{s2:05.2f}]</span> {seg["text"]}</div>\n'
        if not seg_html:
            seg_html = '<div class="seg" style="color:#999">No segments</div>'
        model_cols += f'<div class="model-col"><h4>{mi["name"]}</h4><div class="seg-count">{sp_data["count"]} segs / {sp_data["chars"]} chars</div>{seg_html}</div>\n'
    
    speaker_sections += f'''
    <div class="speaker-card" data-speaker="{speaker}">
      <div class="speaker-header"><h3>{sp_label}</h3>{audio_tag}</div>
      <div class="model-grid">{model_cols}</div>
    </div>'''

# --- Metrics table ---
metrics_rows = ""
for mk in model_keys:
    mi = dashboard_data["models"][mk]
    hc = "bad" if mi["latin_pct"]>5 else "ok" if mi["latin_pct"]>1 else "good"
    metrics_rows += f'<tr><td><strong>{mi["name"]}</strong></td><td>{mi["type"]}</td><td>{mi["total_segments"]}</td><td>{mi["total_chars"]}</td><td class="{hc}">{mi["latin_pct"]}%</td><td>{mi["time"]}s</td></tr>'

html = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Election Committee Analysis Dashboard</title>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;background:#f0f2f5;padding:20px;color:#333}}
  h1{{margin-bottom:5px}} h2{{margin:20px 0 10px;color:#2c3e50}}
  .subtitle{{color:#666;margin-bottom:20px}}
  .kpi-grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:15px;margin-bottom:20px}}
  .kpi-card{{background:#fff;border-radius:8px;padding:20px;text-align:center;box-shadow:0 1px 3px rgba(0,0,0,0.1)}}
  .kpi-label{{font-size:12px;color:#888;text-transform:uppercase;letter-spacing:1px}}
  .kpi-value{{font-size:36px;font-weight:700;margin:8px 0}}
  .kpi-detail{{font-size:12px;color:#aaa}}
  .section-grid{{display:grid;grid-template-columns:1fr 1fr;gap:15px;margin-bottom:20px}}
  .section-card{{background:#fff;border-radius:8px;padding:15px;box-shadow:0 1px 3px rgba(0,0,0,0.1)}}
  .section-card h3{{margin-bottom:10px;color:#2c3e50;font-size:14px}}
  .inner-table{{width:100%;border-collapse:collapse}}
  .inner-table th{{background:#ecf0f1;padding:6px 8px;text-align:left;font-size:12px}}
  .inner-table td{{padding:6px 8px;border-bottom:1px solid #f0f0f0;font-size:13px}}
  .tag{{display:inline-block;background:#3498db;color:#fff;padding:3px 10px;border-radius:12px;margin:3px;font-size:12px}}
  .names-list{{line-height:2}}
  .controls{{background:#fff;padding:15px;border-radius:8px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.1);display:flex;gap:15px;align-items:center}}
  .controls label{{font-weight:600}} .controls select{{padding:6px 12px;border-radius:4px;border:1px solid #ccc;font-size:14px}}
  .full-audio{{background:#fff;padding:15px;border-radius:8px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.1)}}
  .full-audio h3{{margin-bottom:8px}}
  table{{width:100%;border-collapse:collapse;margin-bottom:20px;background:#fff;border-radius:8px;overflow:hidden;box-shadow:0 1px 3px rgba(0,0,0,0.1)}}
  th{{background:#2c3e50;color:#fff;padding:10px;text-align:left}}
  td{{padding:10px;border-bottom:1px solid #eee}} tr:hover{{background:#f8f9fa}}
  .good{{color:#27ae60;font-weight:600}} .ok{{color:#f39c12;font-weight:600}} .bad{{color:#e74c3c;font-weight:600}}
  .speaker-card{{background:#fff;border-radius:8px;padding:20px;margin-bottom:15px;box-shadow:0 1px 3px rgba(0,0,0,0.1)}}
  .speaker-header{{margin-bottom:15px}} .speaker-header h3{{margin-bottom:8px;color:#2c3e50}}
  .model-grid{{display:grid;grid-template-columns:repeat({len(model_keys)},1fr);gap:15px}}
  .model-col{{border:1px solid #e0e0e0;border-radius:6px;padding:12px;background:#fafafa}}
  .model-col h4{{color:#2c3e50;margin-bottom:5px;font-size:13px}}
  .seg-count{{color:#888;font-size:12px;margin-bottom:8px}}
  .seg{{font-size:13px;margin-bottom:4px;line-height:1.4}}
  .ts{{color:#7f8c8d;font-size:11px;font-family:monospace}}
  .hidden{{display:none}}
  .transcript-box{{background:#fff;border-radius:8px;padding:20px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.1);max-height:500px;overflow-y:auto}}
  .transcript-controls{{margin-bottom:15px;display:flex;gap:15px;align-items:center}}
  .transcript-controls label{{font-weight:600}}
  .transcript-controls select{{padding:6px 12px;border-radius:4px;border:1px solid #ccc;font-size:14px}}
  .tseg{{padding:4px 0;border-bottom:1px solid #f5f5f5;font-size:13px;line-height:1.5}}
  .tseg:hover{{background:#f8f9fa}}
  .tseg .ts{{color:#7f8c8d;font-size:11px;font-family:monospace}}
  .tseg-hidden{{display:none}}
</style>
</head><body>
<h1>Election Committee Analysis Dashboard</h1>
<p class="subtitle">Bulgarian speaker diarization + transcription | {len(speakers)} speakers | {AUDIO_DURATION:.0f}s audio</p>

<h2>Key Performance Indicators</h2>
{kpi_section}

<h2>Audio Signal Analysis</h2>
<div class="section-card" style="margin-bottom:20px">
  <h3>Waveform Comparison: Before vs After Filtering</h3>
  <img src="data:image/png;base64,{waveform_b64}" style="width:100%;border-radius:4px" />
</div>
<div class="section-card" style="margin-bottom:20px">
  <h3>Zoom: 0-15s Faint Speech Section</h3>
  <img src="data:image/png;base64,{zoom_b64}" style="width:100%;border-radius:4px" />
</div>

<div class="full-audio">
  <h3>Full Filtered Audio</h3>
  <audio controls style="width:100%"><source src="data:audio/wav;base64,{full_audio_b64}" type="audio/wav"></audio>
</div>

<h2>Model Comparison</h2>
<table>
  <tr><th>Model</th><th>Type</th><th>Segments</th><th>Chars</th><th>Latin %</th><th>Time</th></tr>
  {metrics_rows}
</table>

<div class="controls">
  <label>Filter by speaker:</label>
  <select id="speakerFilter" onchange="filterSpeakers()">
    <option value="all">All Speakers</option>
    {"".join(f'<option value="{sp}">{sp.replace("SPEAKER_","Speaker ")} — {speaker_roles.get(sp,sp)}</option>' for sp in speakers)}
  </select>
</div>

<h2>Unified Transcript</h2>
<div class="transcript-box">
  <div class="transcript-controls">
    <label>Filter:</label>
    <select id="transcriptFilter" onchange="filterTranscript()">
      <option value="all">All Speakers</option>
      {transcript_filter_options}
    </select>
  </div>
  <div id="transcriptBody">{transcript_html}</div>
</div>

<h2>Per-Speaker Transcripts</h2>
<div id="speakerCards">{speaker_sections}</div>

<script>
function filterTranscript(){{
  const val=document.getElementById("transcriptFilter").value;
  document.querySelectorAll(".tseg").forEach(c=>{{
    c.classList.toggle("tseg-hidden",val!=="all"&&c.dataset.speaker!==val);
  }});
}}
function filterSpeakers(){{
  const val=document.getElementById("speakerFilter").value;
  document.querySelectorAll(".speaker-card").forEach(c=>{{
    c.classList.toggle("hidden",val!=="all"&&c.dataset.speaker!==val);
  }});
}}
</script>
</body></html>'''

# Save
dashboard_path = OUTPUT_DIR / "dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(html)
print(f"Dashboard saved: {dashboard_path}")

# Save full results JSON
results_json = {
    "speakers": speakers, "speaker_roles": speaker_roles,
    "diar_segments": diar_segments, "analysis": analysis, "models": {}
}
for mk, mr in merged_results.items():
    results_json["models"][mk] = {
        "name": mr["name"], "time": mr["time"], "type": mr["type"],
        "merged_segments": mr["merged"]
    }
with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    _json.dump(results_json, f, ensure_ascii=False, indent=2)
print(f"Results JSON saved")
print(f"\nOpen dashboard: file://{dashboard_path}")

Dashboard saved: /home/airsrg/work/video_streaming/output/model_comparison/dashboard.html
Results JSON saved

Open dashboard: file:///home/airsrg/work/video_streaming/output/model_comparison/dashboard.html
